# Einstein Summation (einsum) Performance

This notebook demonstrates when `np.einsum` helps vs. hurts performance.

In [1]:
import numpy as np
import time

np.random.seed(42)

def benchmark(funcs: dict, n_runs: int = 100) -> dict:
    """Time multiple implementations and return results in ms."""
    results = {}
    for label, func in funcs.items():
        func()  # warmup
        start = time.perf_counter()
        for _ in range(n_runs):
            func()
        results[label] = (time.perf_counter() - start) / n_runs * 1000
    
    baseline = list(results.values())[0]
    for label, ms in results.items():
        print(f"{label:35s}: {ms:7.3f} ms  ({baseline/ms:5.2f}x)")
    return results

## Example 1: Matrix Multiplication (einsum is slower)

For standard matrix multiplication, NumPy's `@` operator calls highly optimized BLAS libraries. Einsum adds overhead without benefit.

$$C_{ik} = \sum_{j} A_{ij} B_{jk}$$

In [2]:
A = np.random.rand(200, 300)
B = np.random.rand(300, 400)

benchmark({
    "A @ B": lambda: A @ B,
    "einsum('ij,jk->ik', A, B)": lambda: np.einsum('ij,jk->ik', A, B),
})

A @ B                              :   0.092 ms  ( 1.00x)
einsum('ij,jk->ik', A, B)          :   3.306 ms  ( 0.03x)


{'A @ B': 0.0924487499287352, "einsum('ij,jk->ik', A, B)": 3.3059137500822544}

**Takeaway:** For simple operations, use built-in operators.

## Example 2: Sum of Outer Product (einsum is faster)

Computing $\sum_{i,j} a_i b_j$ naively creates a large intermediate array. Einsum avoids this.

$$\text{result} = \sum_{i} \sum_{j} a_i b_j$$

In [3]:
a = np.random.rand(1000)
b = np.random.rand(1000)

benchmark({
    "np.outer(a, b).sum()": lambda: np.outer(a, b).sum(),
    "einsum('i,j->', a, b)": lambda: np.einsum('i,j->', a, b),
})

np.outer(a, b).sum()               :   1.079 ms  ( 1.00x)
einsum('i,j->', a, b)              :   0.069 ms  (15.54x)


{'np.outer(a, b).sum()': 1.0791320801945403,
 "einsum('i,j->', a, b)": 0.06944791995920241}

**Why einsum wins:** The naive approach allocates a 1000×1000 intermediate array, then sums it. Einsum accumulates directly without the intermediate.

## Example 3: Complex Chain (einsum + optimize shines)

When contracting multiple tensors, the order of operations matters enormously. Einsum's `optimize=True` finds the best path.

$$R_{im} = \sum_{j,k,l} W_{ij} X_{jk} Y_{kl} Z_{lm}$$

In [4]:
# Deliberately asymmetric sizes to make contraction order matter
W = np.random.rand(100, 5)
X = np.random.rand(5, 200)
Y = np.random.rand(200, 5)
Z = np.random.rand(5, 100)

benchmark({
    "((W @ X) @ Y) @ Z": lambda: ((W @ X) @ Y) @ Z,
    "W @ (X @ (Y @ Z))": lambda: W @ (X @ (Y @ Z)),
    "einsum (no optimize)": lambda: np.einsum('ij,jk,kl,lm->im', W, X, Y, Z),
    "einsum (optimize=True)": lambda: np.einsum('ij,jk,kl,lm->im', W, X, Y, Z, optimize=True),
}, n_runs=50)

((W @ X) @ Y) @ Z                  :   0.021 ms  ( 1.00x)
W @ (X @ (Y @ Z))                  :   0.030 ms  ( 0.68x)
einsum (no optimize)               : 115.132 ms  ( 0.00x)
einsum (optimize=True)             :   0.044 ms  ( 0.46x)


{'((W @ X) @ Y) @ Z': 0.020535820513032377,
 'W @ (X @ (Y @ Z))': 0.03008417959790677,
 'einsum (no optimize)': 115.13186500000302,
 'einsum (optimize=True)': 0.04428000014740974}

In [ ]:
# Verify all methods give the same result
result1 = ((W @ X) @ Y) @ Z
result2 = W @ (X @ (Y @ Z))
result3 = np.einsum('ij,jk,kl,lm->im', W, X, Y, Z)
result4 = np.einsum('ij,jk,kl,lm->im', W, X, Y, Z, optimize=True)

print("Results match:")
print(f"  Method 1 vs 2: {np.allclose(result1, result2)}")
print(f"  Method 1 vs 3: {np.allclose(result1, result3)}")
print(f"  Method 1 vs 4: {np.allclose(result1, result4)}")
print(f"  Method 3 vs 4: {np.allclose(result3, result4)}")

**Why order matters:**

- Bad order creates large intermediate arrays
- Good order keeps intermediates small
- `optimize=True` uses dynamic programming to find the optimal contraction path

You can inspect the chosen path:

In [5]:
path, info = np.einsum_path('ij,jk,kl,lm->im', W, X, Y, Z, optimize=True)
print(info)

  Complete contraction:  ij,jk,kl,lm->im
         Naive scaling:  5
     Optimized scaling:  3
      Naive FLOP count:  2.000e+08
  Optimized FLOP count:  1.150e+05
   Theoretical speedup:  1739.115
  Largest intermediate:  1.000e+04 elements
--------------------------------------------------------------------------
scaling                  current                                remaining
--------------------------------------------------------------------------
   3                   kl,jk->jl                             ij,lm,jl->im
   3                   jl,ij->li                                lm,li->im
   3                   li,lm->im                                   im->im


## Summary

| Scenario | Recommendation |
|----------|----------------|
| Simple matrix ops | Use `@` operator |
| Avoiding large intermediates | Use `einsum` |
| Multi-tensor contractions | Use `einsum` with `optimize=True` |